In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_20_try_0.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['FuncFormatter']
me:  0
trying: ['all_gaps']
me:  18
trying: ['sample_discourse_id']
me:  2
trying: ['CountVectorizer']
me:  0
trying: ['len_dict']
me:  12
trying: ['plt']
me:  0
trying: ['test_txt']
me:  1
trying: ['warnings']
me:  0
trying: ['train_txt']
me:  1
trying: ['total_gaps']
me:  19
trying: ['train_first']
me:  10
trying: ['train_first_last']
me:  10
trying: ['glob']
me:  0
trying: ['train_last']
me:  10
trying: ['np']
me:  0
trying: ['train']
me:  13
trying: ['tqdm']
me:  0
trying: ['last_ones']
me:  13
trying: ['cols_to_merge']
me:  13
trying: ['print_colored_essay']
me:  21
trying: ['cols_to_display']
me:  14
trying: ['counter']
me:  11
trying: ['add_gap_rows_2']
me:  21
trying: ['IREWR_tmp2']
me:  17
trying: ['IREWR_plug_1']
me:  17
trying: ['pd']
me:  0
trying: ['ax1']
me:  7
trying: ['ax2']
me:  7
trying: ['sample_submission']
me:  1
trying: ['stopwords']
me:  1
trying: ['add_gap_row

me:  12
trying: ['word_dict']
me:  12
trying: ['av_per_essay']
me:  8
trying: ['myword']
me:  12
trying: ['ax']
me:  8
trying: ['data']
me:  12
trying: ['myid']
me:  12
trying: ['t']
me:  12
trying: ['style']
me:  0
Declaring variable FuncFormatter
Declaring variable CountVectorizer
Declaring variable plt
Declaring variable warnings
Declaring variable glob
Declaring variable np
Declaring variable tqdm
Declaring variable pd
Declaring variable style
Declaring variable test_txt
Declaring variable train_txt
Declaring variable sample_submission
Declaring variable stopwords
Declaring variable sample_discourse_id
Declaring variable ax1
Declaring variable ax2
Declaring variable av_per_essay
Declaring variable ax
Declaring variable train_first
Declaring variable train_first_last
Declaring variable train_last
Declaring variable i_1
Declaring variable i_1
Declaring variable counter
Declaring variable len_dict
Declaring variable mylen
Declaring variable txt_file
Declaring variable word_dict
Declar

In [3]:
%%RecordEvent
%%time
### cell 21 ###
# Lowercase discourse text
train['discourse_text'] = train['discourse_text'].str.lower()

# Build stopword list
stop_english = stopwords.words('english')
other_words_to_take_out = ['school', 'students', 'people', 'would', 'could', 'many']
stop_english.extend(other_words_to_take_out)

# Unique discourse types
dt_unique = train['discourse_type'].unique()

# Explode words per discourse type for vectorized processing
df_words = (
    train[['discourse_type']]
         .assign(Word=train['discourse_text'].str.split())
         .explode('Word')
)

# Count total frequencies by discourse_type and Word
counts_all = (
    df_words
      .groupby(['discourse_type', 'Word'], sort=False)
      .size()
      .rename('Frequency')
      .reset_index()
)

# Remove stopwords and take top 10 by descending Frequency
to_keep = ~counts_all['Word'].isin(stop_english)
counts_filtered = counts_all.loc[to_keep]
# Sort by discourse_type (original order) and Frequency descending
counts_sorted = counts_filtered.sort_values(
    ['discourse_type', 'Frequency'], ascending=[True, False]
)
top10 = counts_sorted.groupby('discourse_type', sort=False).head(10)

# Build counts_dict: for each discourse_type, DataFrame indexed by Word, sorted ascending
counts_dict = {
    dt: (
        top10[top10['discourse_type'] == dt]
            [['Word', 'Frequency']]
            .set_index('Word')
            .sort_values(by='Frequency', ascending=True)
    )
    for dt in dt_unique
}

# Preserve df as in original: last discourse_type
df = train.loc[train['discourse_type'] == dt_unique[-1]]

# Keys list
keys = list(counts_dict.keys())

CPU times: user 18 ms, sys: 3.62 ms, total: 21.6 ms
Wall time: 21.6 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_21_try_2.pickle

migration speed (bps): 744776113.8718369
---------------------------
variables to migrate:
ax1 536
ax2 536
IREWR_tmp2 930
counter 28
i_1 28
IREWR_plug_1 61
FuncFormatter 1072
glob 144
CountVectorizer 1072
pd 72
av_per_essay 1338
plt 72
ax 1210
warnings 72
style 72
train_first 302
sample_discourse_id 32
train_txt 126104
train_first_last 334
len_dict 589920
train_last 395
mylen 28
txt_file 208
counts_sorted 214557
word_dict 589920
keys 120
tqdm 1072
myword 28
df_words 618619
data 2813
all_gaps 288
stop_english 1912
myid 61
other_words_to_take_out 104
np 72
t 166
df 827
train 82800
print_colored_essay 144
counts_all 255134
add_gap_rows_2 144
stopwords 48
total_gaps 1329
counts_dict 5455
counts_filtered 214557
cols_to_merge 120
dt_unique 168
cols_to_display 120
i_3 28
top10 10199
last_ones 12567
to_keep 2010
IREWR_tmp 916
add_gap_rows 144
IREWR_plug_2 61
test_txt 120
sample_submission 569
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train', 'stopwords', 'train_txt']
Intermediate variables ['test_txt', 'sample_submission']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['train']
Active variables ['sample_discourse_id']
Intermediate variables []
Future variables ['train', 'stopwords', 'train_txt']
Modified dataframes
======= Cell 2 =======
Input variables ['train']
Active variables ['train', 'cols_to_display']
Intermediate variables []
Future variables ['sample_discourse_id', 'stopwords', 'train_txt']
Modified dataframes
  train
    Input columns: set()
    Changed columns: set()
    Created columns: {'pred_len', 'discourse_len'}
    Deleted columns: set()
======= Cell 3 =======
Input variables ['train', 'cols_to_display']
Active variables []
Intermediate variables []
Future variables ['train', 'train_txt', 'sample_discourse_id', 'cols_to_display', 'stopwords']
Modified dataframes
======= Cell 4 =======
Input variables ['tra

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_21_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[21], f)


In [7]:
opt_output = Out.get(4)

In [ ]:
counts_dict_opt = counts_dict
train_opt = train
df_opt = df
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated_cpu/checkpoints/post_cell_21.pickle
assert counts_dict_opt == counts_dict

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
